## Content: visualize the most travelled roads
Use map-matched data to create a network showing the most travelled paths.

In [1]:
import pandas as pd
import geopandas as gpd
import json
import requests
from shapely.geometry.linestring import LineString, Point
import overpy
import folium
import folium.plugins
import branca
import branca.colormap as cm
import statistics
from collections import Counter
import numpy as np
import requests
import xml.etree.ElementTree as ET
import warnings
warnings.filterwarnings("ignore")

### Naive Approach

In [2]:
df = pd.read_parquet("../data/map_matched_routes.parquet")
df = df[df.matched == True] # only matched routes
df = df.reset_index()

In [3]:
d=[]
for i in range(len(df)):
    d.append(df.unique_id[i][-10:].split('-')[0]+df.unique_id[i][-10:].split('-')[1]+df.unique_id[i][-10:].split('-')[2])
df['dates'] = d
df['dates'] = pd.to_datetime(df['dates'], format='%Y%m%d')

In [4]:
# select month
df['month'] = df['dates'].dt.month
df['year'] = df['dates'].dt.year
df = df[df.month == 5]
df = df[df.year == 2021]
df = df.reset_index()
df = df.drop(['level_0', 'index'], axis=1)
df

,unique_id,route,matched,dates,month,year
0,7b3822b21ad6_2021-05-01,"[[46.094015, 11.118042], [46.094006, 11.118092...",True,2021-05-01,5,2021
1,0000249f-0000-1000-a49f-5b7d6a617b60_2021-05-01,"[[46.071505, 11.124385], [46.071556, 11.12444]...",True,2021-05-01,5,2021
2,00000255-0000-1000-80b5-5b7d6a617b60_2021-05-01,"[[46.06264, 11.130528], [46.062607, 11.130478]...",True,2021-05-01,5,2021
3,00000311-0000-1000-8131-5b7d6a617b60_2021-05-01,"[[46.066735, 11.119816], [46.066664, 11.119142...",True,2021-05-01,5,2021
4,000005a3-0000-1000-82e3-5b7d6a617b60_2021-05-01,"[[46.067347, 11.127076], [46.067383, 11.126828...",True,2021-05-01,5,2021
...,...,...,...,...,...,...
5973,00001f4a-0000-1000-8aaa-5b7d6a617b60_2021-05-31,"[[46.063282, 11.123642], [46.063275, 11.123555...",True,2021-05-31,5,2021
5974,929c30292fc2_2021-05-31,"[[46.071694, 11.121318], [46.072025, 11.121522...",True,2021-05-31,5,2021
5975,2c5de7606219_2021-05-31,"[[46.068837, 11.119139], [46.068787, 11.119144...",True,2021-05-31,5,2021
5976,29d85ecda468_2021-05-31,"[[46.059655, 11.125811], [46.05906, 11.124982]...",True,2021-05-31,5,2021


In [17]:
s=[]
for i in range(len(df)):
    for lsr in range(len(df.route[i])-1):
        segment = ((df.route[i][lsr][1], df.route[i][lsr][0]), (df.route[i][lsr+1][1], df.route[i][lsr+1][0])) # lon-lat
        #segment = LineString(segment)
        s.append(segment)

In [18]:
from collections import Counter
occurrences = Counter(s)
len(occurrences)

23282

In [19]:
max(occurrences.values()), min(occurrences.values()), statistics.mean(occurrences.values()), statistics.median(occurrences.values())

(474, 1, 13.7469289579933, 2.0)

In [20]:
f_map = folium.Map(location=[46.066,11.133], tiles="OpenStreetMap", zoom_start=12)


colormap = cm.LinearColormap(colors=['darkblue', 'blue', 'cyan', 'yellow', 'orange', 'red'],
                            index = np.linspace(min(occurrences.values()), max(occurrences.values()), num=6),
                            vmin = min(occurrences.values()), vmax = max(occurrences.values()), 
                            caption='Number of times the segment was passed by')


fg = folium.FeatureGroup(name='naive')    

for lstr in occurrences.keys(): 
    # fg = folium.FeatureGroup(name=str(df.month[i])) # a feature group for each month
    color = colormap(occurrences[lstr])
    w = occurrences[lstr]/50
    lstr = [[lstr[0][1], lstr[0][0]], [lstr[1][1], lstr[1][0]]]
    folium.vector_layers.PolyLine(lstr, color=color, weight=w).add_to(fg)


f_map.add_child(fg)
f_map.add_child(colormap)

f_map.save('naive_map_tratte_forti.html')

### Better solution
1. Use map-matched data and map-match the route again with Valhalla docker using `trace_attributes` mode and the using as `shape_match` the `edge_walk` algorithm. According to the [documentation](https://valhalla.readthedocs.io/en/latest/api/map-matching/api-reference/#shape-matching-parameters), this algorithm should only be used when the input route is already map-matched or at least is precise enough to be really closed to the OSM ways.

2. From the response, extract the way ids. Each route will be assigned to a list of ways ids, corresponding to the segment of the street network that route crosses.

3. Get the count of how many times each segment appears, using the way ids. Finally, use these ids to query the [Overpass API](https://overpass-turbo.eu/#) to get the matching coordinates of the corresponding segments and plot it on the map according to their frequency.  The documentation and basic examples are available at this [link](https://python-overpy.readthedocs.io/en/latest/example.html#ways). To avoid running into _OverpassTooManyRequests_, the [docker image](https://github.com/wiktorn/Overpass-API) for the Overpass API was used.

In [47]:
df

,unique_id,route,matched,dates,month,year
0,7b3822b21ad6_2021-05-01,"[[46.094015, 11.118042], [46.094006, 11.118092...",True,2021-05-01,5,2021
1,0000249f-0000-1000-a49f-5b7d6a617b60_2021-05-01,"[[46.071505, 11.124385], [46.071556, 11.12444]...",True,2021-05-01,5,2021
2,00000255-0000-1000-80b5-5b7d6a617b60_2021-05-01,"[[46.06264, 11.130528], [46.062607, 11.130478]...",True,2021-05-01,5,2021
3,00000311-0000-1000-8131-5b7d6a617b60_2021-05-01,"[[46.066735, 11.119816], [46.066664, 11.119142...",True,2021-05-01,5,2021
4,000005a3-0000-1000-82e3-5b7d6a617b60_2021-05-01,"[[46.067347, 11.127076], [46.067383, 11.126828...",True,2021-05-01,5,2021
...,...,...,...,...,...,...
5973,00001f4a-0000-1000-8aaa-5b7d6a617b60_2021-05-31,"[[46.063282, 11.123642], [46.063275, 11.123555...",True,2021-05-31,5,2021
5974,929c30292fc2_2021-05-31,"[[46.071694, 11.121318], [46.072025, 11.121522...",True,2021-05-31,5,2021
5975,2c5de7606219_2021-05-31,"[[46.068837, 11.119139], [46.068787, 11.119144...",True,2021-05-31,5,2021
5976,29d85ecda468_2021-05-31,"[[46.059655, 11.125811], [46.05906, 11.124982]...",True,2021-05-31,5,2021


In [5]:
edge_ids=[]
ids = df.unique_id.to_list()
for id in range(len(ids)):
    df_temp = df[df.unique_id==ids[id]]
    df_points = pd.DataFrame({'lon':[el[1] for i in df_temp.route for el in i], 'lat':[el[0] for i in df_temp.route for el in i]})

    # request
    meili_coordinates = df_points.to_json(orient='records')
    meili_head = '{"shape":'
    meili_tail = ""","search_radius": 300, "shape_match":"edge_walk", "costing":"bicycle", "format":"osrm"}""" # use 'shape_match' : 'edge_walk'
    meili_request_body = meili_head + meili_coordinates + meili_tail
    url = "http://localhost:8002/trace_attributes" # use /trace_attributes 
    headers = {'Content-type': 'application/json'}
    data = str(meili_request_body)

    r = requests.post(url, data=data, headers=headers)
    
    response_text = json.loads(r.text)

    if r.status_code == 200:
        l=[]
        for i in range(len(response_text['edges'])):
            l.append(response_text['edges'][i]['way_id']) #, response_text['osm_changeset'])

        edge_ids.append(list(set(l)))
    else:
        edge_ids.append([])  

In [48]:
df['edge_ids'] = edge_ids

In [49]:
df = df[df['edge_ids'].map(lambda d: len(d)) > 0] # remove rows where edge_ids is empty

In [6]:
c = []
for el in edge_ids:
    c.extend(el)
c = Counter(c)
c

Counter({170076129: 11,
         307993862: 21,
         170076124: 12,
         25874135: 9,
         25874489: 25,
         170076123: 5,
         97111260: 24,
         170076127: 6,
         80410756: 10,
         23952134: 61,
         240092039: 79,
         222748808: 55,
         80410759: 9,
         97084819: 35,
         229137174: 137,
         216745509: 226,
         150886566: 32,
         770864809: 28,
         127209003: 276,
         68286129: 84,
         23857969: 82,
         223023923: 131,
         150886581: 32,
         150886597: 79,
         24581836: 59,
         150886604: 51,
         23858004: 147,
         150886613: 44,
         150886616: 37,
         150886618: 74,
         227756257: 231,
         144843361: 50,
         150886625: 43,
         150886629: 31,
         144843368: 80,
         754578288: 145,
         80410751: 10,
         24916544: 41,
         24916515: 61,
         88597865: 45,
         24449618: 33,
         24443798: 41,
      

_________________________________
Test Overpass API

In [157]:
import overpy
api = overpy.Overpass()
result = api.query("""[out:json];
                    way(170076129);
                    out geom;""")

In [162]:
len(result.ways), result.ways, result.ways[0].get_nodes(resolve_missing=True)

(1,
 [<overpy.Way id=170076129 nodes=[907823976, 907823963, 907823962, 4786649726, 1271454530, 907823964, 907823975, 907823965]>],
 [<overpy.Node id=907823976 lat=46.0940056 lon=11.1180917>,
  <overpy.Node id=907823963 lat=46.0940201 lon=11.1180161>,
  <overpy.Node id=907823962 lat=46.0940662 lon=11.1179733>,
  <overpy.Node id=4786649726 lat=46.0945524 lon=11.1177386>,
  <overpy.Node id=1271454530 lat=46.0950002 lon=11.1174901>,
  <overpy.Node id=907823964 lat=46.0950538 lon=11.1174607>,
  <overpy.Node id=907823975 lat=46.0951031 lon=11.1174639>,
  <overpy.Node id=907823965 lat=46.0951444 lon=11.1175226>])

In [165]:
result.ways[0].get_nodes(resolve_missing=True)[0].lat

Decimal('46.0940056')

End Test

_________________________________

In [44]:
# ask for all way ids in a single request to avoid OverpassTooManyRequests error
els = 'id: '
for el in c.keys():
    els = els + str(el) +', '
els = els[:-2] # string of way ids to pass to request coordinates

# using overpy:
# api = overpy.Overpass()
# result = api.query("""[out:json];
#                     way("""+str(els)+""");
#                     out geom;""")

# using Overpass Docker API:
# https://github.com/wiktorn/Overpass-API

url = 'http://localhost:12345/api/interpreter'
data = 'data=way('+str(els)+');out geom;'
r = requests.post(url, data=data)
root = ET.fromstring(r.text)

nodes_coords = []
waysids = []
for child in root:
    if child.tag == 'way':
        waysids.append(int(child.attrib['id']))
        nodes=[]
        for node in child:
            if node.tag == 'nd':
                # print(node.attrib['ref']) # node id
                nodes.append([float(node.attrib['lat']), float(node.attrib['lon'])])
        nodes_coords.append(nodes)

In [45]:
cnt=[]
for id in waysids:
    cnt.append(c[id])

In [46]:
ways = pd.DataFrame({'way_id': waysids, 'route': nodes_coords, 'freq': cnt})
ways

,way_id,route,freq
0,22986084,"[[46.0666524, 11.1190333], [46.0666403, 11.118...",349
1,22986312,"[[46.0674662, 11.1266008], [46.0675156, 11.126...",190
2,22986560,"[[46.069021, 11.1225351], [46.0693306, 11.1224...",117
3,23056771,"[[46.0969958, 11.1125195], [46.096235, 11.1127...",5
4,23058066,"[[46.0838565, 11.1146839], [46.0839814, 11.114...",19
...,...,...,...
2487,1058552722,"[[46.097594, 11.1024637], [46.0973823, 11.1026...",1
2488,1058552723,"[[46.1010479, 11.1017295], [46.1009174, 11.101...",2
2489,1059033164,"[[46.0996549, 11.117578], [46.099468, 11.11766...",3
2490,1059175213,"[[46.058549, 11.1227312], [46.0585422, 11.1227...",49


In [100]:
text_html = '''
{% macro html(this, kwargs) %}
<div style="
    position: fixed; 
    top: 50px;
    right: 10px;
    width: 360px;
    height: 50px; 
    z-index:9998;
    font-size:14px;
    ">
    <p style="color:white;font-size:90%;margin-right:10px;">Number of times the segment was passed through</p>
</div>
{% endmacro %}
'''

In [103]:
f_map = folium.Map(location=[46.066,11.133], tiles='cartodbdark_matter', zoom_start=14) #  tiles="cartodbdark_matter" "Stamen Toner" "OpenStreetMap"


colormap = cm.LinearColormap(colors=['cyan', 'yellow', 'orange', 'red'], # ['darkblue', 'blue', 'cyan', 'yellow', 'orange', 'red']
                            index = np.linspace(min(c.values()), max(c.values()), num=4),
                            vmin = min(c.values()), vmax = max(c.values()) ) # caption='Number of times the segment was passed through'

text = folium.MacroElement().add_to(f_map) 
text._template = branca.element.Template(text_html)

fg = folium.FeatureGroup(name='overpass_tratte_forti')    


for i in range(len(ways)): 
    # fg = folium.FeatureGroup(name=str(df.month[i])) # a feature group for each month
    color = colormap(c[ways.way_id[i]])
    w = min(ways['freq'][i]/100 , 5)
    folium.vector_layers.PolyLine(ways.route[i], color=color, weight=w).add_to(fg)


f_map.add_child(fg)
f_map.add_child(colormap)

f_map.save('html/overpass_tratte_forti.html')

### Heatmap by hour of a given day

In [ ]:
df = pd.read_parquet("../data/trips_pointv3_cleaned.parquet")
df.point_latitude = df.point_latitude.astype(float)
df.point_longitude = df.point_longitude.astype(float)
df['date'] = pd.to_datetime(df.date)
df['hour'] = df['point_timestamp'].dt.hour

In [ ]:
# select one day
df = df[df.date == pd.Timestamp(2022, 1, 31)]

In [ ]:
lat_long_list = []
for i in range(1,25): 
    temp=[]
    for index, instance in df[df['hour'] == i].iterrows():
        temp.append([instance['point_latitude'],instance['point_longitude']])
    lat_long_list.append(temp)

In [ ]:
import folium
from folium.plugins import HeatMapWithTime
from branca.element import Figure

fig = Figure(width=850, height=550)
m = folium.Map(location=[46.06, 11.13], zoom_start=12, control_scale=True)
fig.add_child(m)
HeatMapWithTime(lat_long_list, radius=5, auto_play=True, position='bottomright').add_to(m)
m.save('heatmap_by_hour.html')

### Heatmap through a month

In [ ]:
df = pd.read_parquet("../data/trips_pointv3_cleaned.parquet")
df.point_latitude = df.point_latitude.astype(float)
df.point_longitude = df.point_longitude.astype(float)
df['date'] = pd.to_datetime(df.date)
df['hour'] = df['point_timestamp'].dt.hour

In [ ]:
# select month
df['month'] = df['point_timestamp'].dt.month
df['year'] = df['point_timestamp'].dt.year
df = df[df.month == 2]
df = df[df.year == 2022]

In [ ]:
lat_long_list = []
for i in df['point_timestamp'].unique():
    temp=[]
    for index, instance in df[df['point_timestamp'] == i].iterrows():
        temp.append([instance['point_latitude'],instance['point_longitude']])
    lat_long_list.append(temp)

In [ ]:
# time indexes for map
time_index = []
for i in df['point_timestamp'].unique():
    time_index.append(i)

date_strings = [d.strftime('%d/%m/%Y, %H:%M:%S') for d in time_index]

In [ ]:
m = folium.Map(location=[46.06, 11.13], zoom_start = 12, control_scale=True)
HeatMapWithTime(lat_long_list, radius=5, auto_play=True, position='bottomright', name="cluster", index = date_strings, max_opacity=0.7).add_to(m)

m.save('heatmap_month.html')